# Legal QA Evaluation

Notebook này đánh giá hệ thống Multi-Agent RAG theo dataset `data/evaluation/legal_qa_eval_30.jsonl`.

Các lớp đánh giá chính:
- **Retrieval**: tìm đúng văn bản / điều / khoản / điểm / level.
- **Answer**: bao phủ facts, không chứa forbidden terms, citation hiển thị đúng `[1]`, `[2]`.
- **Policy**: xử lý `grounded`, `insufficient_data`, `out_of_scope`.

Mặc định notebook chỉ chạy retrieval vì rẻ và không cần gọi LLM. Bật `RUN_ANSWER=True` hoặc `RUN_GRAPH=True` khi muốn đánh giá sinh câu trả lời.

In [1]:
from __future__ import annotations

import json
import os
import re
import sys
import time
from datetime import datetime
from pathlib import Path
from statistics import mean
from typing import Any

try:
    import pandas as pd
except Exception:
    pd = None

def find_repo_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for path in [start, *start.parents]:
        if (path / 'src').exists() and (path / 'data').exists():
            return path
    raise RuntimeError('Cannot find repo root')

ROOT_DIR = find_repo_root()
sys.path.insert(0, str(ROOT_DIR))
os.chdir(ROOT_DIR)

DATASET_PATH = ROOT_DIR / 'data' / 'evaluation' / 'legal_qa_eval_30.jsonl'
REPORT_DIR = ROOT_DIR / 'eval_reports'
REPORT_DIR.mkdir(exist_ok=True)

# Controls. Keep answer/graph off by default to avoid LLM cost.
TOP_K = 5
RUN_ANSWER = False
RUN_GRAPH = False

print('ROOT_DIR =', ROOT_DIR)
print('DATASET_PATH =', DATASET_PATH)

ROOT_DIR = D:\source\Multi-Agent-RAG-for-Vietnamese-Legal-QA
DATASET_PATH = D:\source\Multi-Agent-RAG-for-Vietnamese-Legal-QA\data\evaluation\legal_qa_eval_30.jsonl


In [2]:
def load_jsonl(path: Path) -> list[dict[str, Any]]:
    rows = []
    with path.open('r', encoding='utf-8') as file:
        for line_no, line in enumerate(file, 1):
            line = line.strip()
            if not line:
                continue
            try:
                rows.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON at line {line_no}: {exc}') from exc
    return rows

cases = load_jsonl(DATASET_PATH)
print(f'Loaded {len(cases)} evaluation cases')

if pd:
    display(pd.DataFrame(cases)[['id', 'category', 'type', 'answer_policy', 'question']].head(10))
else:
    cases[:3]

Loaded 32 evaluation cases


## Metric Helpers

Các hàm dưới đây chấm bằng rule/metadata, không dùng LLM judge. Đây là lớp đánh giá ổn định nhất cho legal QA.

In [3]:
def normalize_text(text: str) -> str:
    text = (text or '').lower()
    text = text.replace(',', '.')
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def contains_fact(answer: str, fact: str) -> bool:
    return normalize_text(fact) in normalize_text(answer)

def display_citation_ids(answer: str) -> set[int]:
    return {int(match.group(1)) for match in re.finditer(r'\[(\d+)\]', answer or '')}

def internal_source_ids_in_answer(answer: str) -> set[str]:
    return {
        match.group(1).replace(' ', '')
        for match in re.finditer(r'\[(S\d+|Web\s+\d+)\]', answer or '', flags=re.IGNORECASE)
    }

def doc_meta(doc: dict[str, Any]) -> dict[str, Any]:
    return doc.get('metadata', {}) or {}

def hit_any(values: list[Any], expected: list[Any]) -> bool:
    if not expected:
        return True
    return any(value in expected for value in values)

def reciprocal_rank(docs: list[dict[str, Any]], expected_doc_ids: list[str]) -> float:
    if not expected_doc_ids:
        return 0.0
    for idx, doc in enumerate(docs, 1):
        if doc_meta(doc).get('doc_id') in expected_doc_ids:
            return 1.0 / idx
    return 0.0

def score_retrieval(case: dict[str, Any], docs: list[dict[str, Any]], top_k: int = TOP_K) -> dict[str, Any]:
    top_docs = docs[:top_k]
    metas = [doc_meta(doc) for doc in top_docs]
    doc_ids = [m.get('doc_id') for m in metas]
    articles = [m.get('article_number') for m in metas]
    clauses = [m.get('clause_number') for m in metas]
    points = [m.get('point_label') for m in metas]
    levels = [m.get('level') for m in metas]

    expected_doc_ids = case.get('expected_doc_ids') or []
    expected_articles = case.get('expected_articles') or []
    expected_clauses = case.get('expected_clauses') or []
    expected_points = case.get('expected_points') or []
    expected_level = case.get('expected_level') or ''

    return {
        'doc_hit': hit_any(doc_ids, expected_doc_ids) if expected_doc_ids else True,
        'article_hit': hit_any(articles, expected_articles),
        'clause_hit': hit_any(clauses, expected_clauses),
        'point_hit': hit_any(points, expected_points),
        'level_hit': (expected_level in levels) if expected_level else True,
        'mrr': reciprocal_rank(top_docs, expected_doc_ids),
        'retrieved_doc_ids': doc_ids,
        'retrieved_articles': articles,
        'retrieved_clauses': clauses,
        'retrieved_points': points,
        'retrieved_levels': levels,
    }

def score_answer(case: dict[str, Any], answer: str, citations: list[dict[str, Any]]) -> dict[str, Any]:
    expected_facts = case.get('expected_facts') or []
    must_not_include = case.get('must_not_include') or []
    missing_facts = [fact for fact in expected_facts if not contains_fact(answer, fact)]
    forbidden_hits = [term for term in must_not_include if contains_fact(answer, term)]

    display_ids = display_citation_ids(answer)
    citation_display_ids = {
        int(c.get('display_id'))
        for c in citations
        if isinstance(c, dict) and c.get('display_id') is not None
    }
    citation_source_ids = [c.get('source_id') for c in citations if isinstance(c, dict)]
    citation_urls = [c.get('url') for c in citations if isinstance(c, dict)]
    citation_doc_ids = [
        (c.get('metadata') or {}).get('doc_id')
        for c in citations
        if isinstance(c, dict)
    ]

    must_cite = bool(case.get('must_cite'))
    must_have_url = bool(case.get('must_have_citation_url'))
    expected_doc_ids = set(case.get('expected_doc_ids') or [])

    citation_present = bool(display_ids or citations)
    display_citation_valid = not internal_source_ids_in_answer(answer) and (not must_cite or bool(display_ids))
    citation_mapping_valid = display_ids.issubset(citation_display_ids) if display_ids else not must_cite
    citation_has_source_id = all(bool(source_id) for source_id in citation_source_ids) if citations else not must_cite
    citation_url_present = any(citation_urls) if must_have_url else True
    citation_doc_match = bool(expected_doc_ids.intersection(set(citation_doc_ids))) if expected_doc_ids and citations else not expected_doc_ids

    fact_coverage = 1.0 if not expected_facts else (len(expected_facts) - len(missing_facts)) / len(expected_facts)

    return {
        'fact_coverage': fact_coverage,
        'missing_facts': missing_facts,
        'forbidden_hits': forbidden_hits,
        'forbidden_ok': not forbidden_hits,
        'citation_present': citation_present if must_cite else True,
        'display_citation_valid': display_citation_valid,
        'citation_mapping_valid': citation_mapping_valid,
        'citation_has_source_id': citation_has_source_id,
        'citation_url_present': citation_url_present,
        'citation_doc_match': citation_doc_match,
        'display_ids': sorted(display_ids),
        'citation_display_ids': sorted(citation_display_ids),
        'citation_source_ids': citation_source_ids,
        'citation_doc_ids': citation_doc_ids,
    }

def policy_success(case: dict[str, Any], answer: str, retrieval_score: dict[str, Any], answer_score: dict[str, Any] | None = None) -> bool:
    policy = case.get('answer_policy')
    normalized = normalize_text(answer)
    if policy == 'grounded':
        return bool(retrieval_score.get('doc_hit')) and (answer_score or {}).get('forbidden_ok', True)
    if policy == 'insufficient_data':
        refusal_terms = ['không đủ dữ liệu', 'chưa đủ dữ liệu', 'không tìm thấy', 'không có căn cứ', 'chưa có căn cứ']
        return any(term in normalized for term in refusal_terms) or not retrieval_score.get('doc_hit')
    if policy == 'out_of_scope':
        refusal_terms = ['ngoài phạm vi', 'không phải câu hỏi pháp lý', 'không hỗ trợ']
        return any(term in normalized for term in refusal_terms) or not retrieval_score.get('doc_hit')
    return True

def bool_mean(rows: list[dict[str, Any]], key: str) -> float:
    values = [bool(row.get(key)) for row in rows if key in row]
    return mean(values) if values else 0.0

def float_mean(rows: list[dict[str, Any]], key: str) -> float:
    values = [float(row.get(key, 0.0)) for row in rows if key in row]
    return mean(values) if values else 0.0

## Run Retrieval Evaluation

In [4]:
from src.graph.state import create_initial_state
from src.agents.retriever import retriever_node
from src.graph.runtime_store import get_documents, get_citations

def run_retrieval_case(case: dict[str, Any]) -> dict[str, Any]:
    state = create_initial_state(case['question'])
    started = time.perf_counter()
    updates = retriever_node(state)
    state.update(updates)
    docs = get_documents(state)
    elapsed_ms = int((time.perf_counter() - started) * 1000)
    score = score_retrieval(case, docs, TOP_K)
    return {
        'id': case['id'],
        'category': case['category'],
        'type': case['type'],
        'answer_policy': case['answer_policy'],
        'question': case['question'],
        'retrieval_error': updates.get('error'),
        'retrieval_ms': elapsed_ms,
        **score,
    }

retrieval_results = [run_retrieval_case(case) for case in cases]
retrieval_summary = {
    'cases': len(retrieval_results),
    'doc_hit@k': bool_mean(retrieval_results, 'doc_hit'),
    'article_hit@k': bool_mean(retrieval_results, 'article_hit'),
    'clause_hit@k': bool_mean(retrieval_results, 'clause_hit'),
    'point_hit@k': bool_mean(retrieval_results, 'point_hit'),
    'level_hit@k': bool_mean(retrieval_results, 'level_hit'),
    'mrr': float_mean(retrieval_results, 'mrr'),
    'avg_retrieval_ms': float_mean(retrieval_results, 'retrieval_ms'),
}
retrieval_summary

d:\source\Multi-Agent-RAG-for-Vietnamese-Legal-QA\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


2026-05-31 11:13:29 | INFO     | src.agents.web_searcher:<module>:22 | [WEB_SEARCHER] Using TavilySearch from langchain_tavily
2026-05-31 11:13:29 | INFO     | src.agents.retriever:retriever_node:359 | Retrieving documents for question: Người lao động có những quyền cơ bản nào theo Bộ luật Lao động?...
2026-05-31 11:13:29 | INFO     | src.utils.embedding:_get_model:54 | [EMBEDDING] Đang khởi tạo model 'paraphrase-multilingual-MiniLM-L12-v2' trên device='cpu'...
2026-05-31 11:13:29 | INFO     | src.utils.embedding:_get_model:61 | [EMBEDDING] LƯU Ý: Nếu chạy lần đầu, quá trình này có thể mất vài phút để tải model. Các lần sau sẽ tự động dùng bản cache trên máy.


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2338.66it/s]
BertModel LOAD REPORT from: sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


2026-05-31 11:13:37 | INFO     | src.utils.embedding:_get_model:77 | [EMBEDDING] Model đã tải xong. Embedding dimension: 384
2026-05-31 11:13:37 | INFO     | src.data_pipeline.indexer:get_qdrant_client:118 | [INDEXER] Connected to Qdrant http://localhost:6333
2026-05-31 11:13:37 | INFO     | src.agents.retriever:retriever_node:388 | [RETRIEVER] Retrieved 5 fused candidates with filters={'trang_thai': 'Hiện hành'} preferences={'preferred_levels': [], 'preferred_doc_id_keywords': [], 'wants_table': False, 'wants_appendix': False, 'wants_land_price': False}
2026-05-31 11:13:37 | INFO     | src.agents.retriever:_expand_context:341 | [RETRIEVER] Expanded context 5 -> 15 chunks
2026-05-31 11:13:37 | INFO     | src.agents.retriever:retriever_node:413 | Successfully formatted 15 documents
2026-05-31 11:13:37 | INFO     | src.agents.retriever:retriever_node:359 | Retrieving documents for question: Người sử dụng lao động không được làm gì khi giao kết hoặc thực hiện hợp đồng lao động?...
2026-05

{'cases': 32,
 'doc_hit@k': 1,
 'article_hit@k': 0.9375,
 'clause_hit@k': 0.84375,
 'point_hit@k': 1,
 'level_hit@k': 1,
 'mrr': 0.90625,
 'avg_retrieval_ms': 416.875}

In [5]:
retrieval_df = pd.DataFrame(retrieval_results) if pd else retrieval_results
if pd:
    display(retrieval_df[['id', 'category', 'doc_hit', 'article_hit', 'clause_hit', 'level_hit', 'mrr', 'retrieved_doc_ids', 'retrieved_articles']])
else:
    retrieval_results[:3]

retrieval_failures = [r for r in retrieval_results if not (r['doc_hit'] and r['article_hit'] and r['clause_hit'] and r['point_hit'] and r['level_hit'])]
print(f'Retrieval failures: {len(retrieval_failures)}')
if pd and retrieval_failures:
    display(pd.DataFrame(retrieval_failures)[['id', 'question', 'retrieved_doc_ids', 'retrieved_articles', 'retrieved_clauses', 'retrieved_levels']])

Retrieval failures: 7


## Run Answer Evaluation

Bật `RUN_ANSWER=True` ở cell đầu để gọi generator. Cell này có thể tốn Gemini quota.

In [6]:
RUN_ANSWER = True

cases = load_jsonl(DATASET_PATH)[:5]

In [7]:
from src.agents.generator import generator_node

def run_answer_case(case: dict[str, Any]) -> dict[str, Any]:
    state = create_initial_state(case['question'])
    retrieval_started = time.perf_counter()
    retrieval_updates = retriever_node(state)
    state.update(retrieval_updates)
    docs = get_documents(state)
    retrieval_ms = int((time.perf_counter() - retrieval_started) * 1000)
    retrieval_score = score_retrieval(case, docs, TOP_K)

    generation_started = time.perf_counter()
    generation_updates = generator_node(state)
    state.update(generation_updates)
    generation_ms = int((time.perf_counter() - generation_started) * 1000)
    answer = state.get('answer') or ''
    citations = get_citations(state)
    answer_score = score_answer(case, answer, citations)

    return {
        'id': case['id'],
        'category': case['category'],
        'type': case['type'],
        'answer_policy': case['answer_policy'],
        'question': case['question'],
        'answer': answer,
        'answer_preview': answer[:500],
        'retrieval_ms': retrieval_ms,
        'generation_ms': generation_ms,
        'retrieval_error': retrieval_updates.get('error'),
        'generation_error': generation_updates.get('error'),
        **retrieval_score,
        **answer_score,
        'policy_success': policy_success(case, answer, retrieval_score, answer_score),
    }

answer_results = []
if RUN_ANSWER:
    answer_results = [run_answer_case(case) for case in cases]
    answer_summary = {
        'cases': len(answer_results),
        'doc_hit@k': bool_mean(answer_results, 'doc_hit'),
        'article_hit@k': bool_mean(answer_results, 'article_hit'),
        'fact_coverage': float_mean(answer_results, 'fact_coverage'),
        'forbidden_ok': bool_mean(answer_results, 'forbidden_ok'),
        'citation_present': bool_mean(answer_results, 'citation_present'),
        'display_citation_valid': bool_mean(answer_results, 'display_citation_valid'),
        'citation_mapping_valid': bool_mean(answer_results, 'citation_mapping_valid'),
        'citation_url_present': bool_mean(answer_results, 'citation_url_present'),
        'citation_doc_match': bool_mean(answer_results, 'citation_doc_match'),
        'policy_success': bool_mean(answer_results, 'policy_success'),
        'avg_retrieval_ms': float_mean(answer_results, 'retrieval_ms'),
        'avg_generation_ms': float_mean(answer_results, 'generation_ms'),
    }
else:
    answer_summary = {'skipped': True, 'reason': 'Set RUN_ANSWER=True to run generator evaluation.'}

answer_summary

2026-05-31 11:19:57 | INFO     | src.agents.retriever:retriever_node:359 | Retrieving documents for question: Người lao động có những quyền cơ bản nào theo Bộ luật Lao động?...
2026-05-31 11:19:57 | INFO     | src.data_pipeline.indexer:get_qdrant_client:118 | [INDEXER] Connected to Qdrant http://localhost:6333
2026-05-31 11:19:58 | INFO     | src.agents.retriever:retriever_node:388 | [RETRIEVER] Retrieved 5 fused candidates with filters={'trang_thai': 'Hiện hành'} preferences={'preferred_levels': [], 'preferred_doc_id_keywords': [], 'wants_table': False, 'wants_appendix': False, 'wants_land_price': False}
2026-05-31 11:19:58 | INFO     | src.agents.retriever:_expand_context:341 | [RETRIEVER] Expanded context 5 -> 15 chunks
2026-05-31 11:19:58 | INFO     | src.agents.retriever:retriever_node:413 | Successfully formatted 15 documents
2026-05-31 11:19:58 | INFO     | src.agents.generator:generator_node:489 | Generating answer...
2026-05-31 11:19:58 | INFO     | src.agents.generator:genera

{'cases': 5,
 'doc_hit@k': 1,
 'article_hit@k': 0.8,
 'fact_coverage': 0.7,
 'forbidden_ok': 1,
 'citation_present': 1,
 'display_citation_valid': 1,
 'citation_mapping_valid': 1,
 'citation_url_present': 1,
 'citation_doc_match': 1,
 'policy_success': 1,
 'avg_retrieval_ms': 96.4,
 'avg_generation_ms': 9944.0}

In [15]:
if RUN_ANSWER and pd:
    answer_df = pd.DataFrame(answer_results)
    cols = ['id', 'category', 'fact_coverage', 'forbidden_ok', 'display_citation_valid', 'citation_mapping_valid', 'citation_doc_match', 'policy_success', 'missing_facts', 'forbidden_hits']
    display(answer_df[cols])
    failures = answer_df[(answer_df['fact_coverage'] < 1.0) | (~answer_df['forbidden_ok']) | (~answer_df['display_citation_valid']) | (~answer_df['policy_success'])]
    print(f'Answer failures: {len(failures)}')
    if len(failures):
        display(failures[['id', 'question', 'missing_facts', 'forbidden_hits', 'answer_preview']])
elif not RUN_ANSWER:
    print('Answer evaluation skipped.')

,id,category,fact_coverage,forbidden_ok,display_citation_valid,citation_mapping_valid,citation_doc_match,policy_success,missing_facts,forbidden_hits
0,labor_001,labor,0.500000,True,True,True,True,True,"[tự do lựa chọn việc làm, hưởng lương, đơn phư...",[]
1,labor_002,labor,0.333333,True,True,True,True,True,"[yêu cầu biện pháp bảo đảm bằng tiền, buộc ngư...",[]
2,labor_003,labor,0.666667,True,True,True,True,True,[trừ trường hợp],[]
3,labor_004,labor,1.000000,True,True,True,True,True,[],[]
4,labor_005,labor,1.000000,True,True,True,True,True,[],[]


Answer failures: 3


,id,question,missing_facts,forbidden_hits,answer_preview
0,labor_001,Người lao động có những quyền cơ bản nào theo ...,"[tự do lựa chọn việc làm, hưởng lương, đơn phư...",[],"Theo Bộ luật Lao động, người lao động có các q..."
1,labor_002,Người sử dụng lao động không được làm gì khi g...,"[yêu cầu biện pháp bảo đảm bằng tiền, buộc ngư...",[],"Khi giao kết hoặc thực hiện hợp đồng lao động,..."
2,labor_003,Hợp đồng lao động có thể giao kết bằng lời nói...,[trừ trường hợp],[],Hợp đồng lao động có thể được giao kết bằng lờ...


In [16]:
failures = answer_df[
    (answer_df["fact_coverage"] < 1.0)
    | (~answer_df["article_hit"])
    | (~answer_df["forbidden_ok"])
    | (~answer_df["display_citation_valid"])
    | (~answer_df["policy_success"])
]
display(failures[["id", "question", "missing_facts", "retrieved_articles", "answer_preview"]])

,id,question,missing_facts,retrieved_articles,answer_preview
0,labor_001,Người lao động có những quyền cơ bản nào theo ...,"[tự do lựa chọn việc làm, hưởng lương, đơn phư...","[100, 117, 178, 58, 186]","Theo Bộ luật Lao động, người lao động có các q..."
1,labor_002,Người sử dụng lao động không được làm gì khi g...,"[yêu cầu biện pháp bảo đảm bằng tiền, buộc ngư...","[17, 28, 31, 18, 22]","Khi giao kết hoặc thực hiện hợp đồng lao động,..."
2,labor_003,Hợp đồng lao động có thể giao kết bằng lời nói...,[trừ trường hợp],"[14, 13, 33, 162, 28]",Hợp đồng lao động có thể được giao kết bằng lờ...


## Optional Full Graph Evaluation

Bật `RUN_GRAPH=True` để chạy toàn bộ LangGraph gồm router, grader, web search, generator, hallucination grader. Đây là mode gần production nhất nhưng tốn chi phí và thời gian nhất.

In [ ]:
def run_graph_case(case: dict[str, Any]) -> dict[str, Any]:
    from src.graph.graph import app as graph_app
    state = create_initial_state(case['question'])
    started = time.perf_counter()
    final_state = graph_app.invoke(state)
    elapsed_ms = int((time.perf_counter() - started) * 1000)
    docs = get_documents(final_state)
    citations = get_citations(final_state)
    answer = final_state.get('answer') or ''
    retrieval_score = score_retrieval(case, docs, TOP_K)
    answer_score = score_answer(case, answer, citations)
    return {
        'id': case['id'],
        'category': case['category'],
        'type': case['type'],
        'answer_policy': case['answer_policy'],
        'question': case['question'],
        'answer': answer,
        'answer_preview': answer[:500],
        'graph_ms': elapsed_ms,
        'intent': final_state.get('intent'),
        'grader_verdict': final_state.get('grader_verdict'),
        'hallucination_verdict': final_state.get('hallucination_verdict'),
        'generation_attempt': final_state.get('generation_attempt'),
        'error': final_state.get('error'),
        **retrieval_score,
        **answer_score,
        'policy_success': policy_success(case, answer, retrieval_score, answer_score),
    }

graph_results = []
if RUN_GRAPH:
    graph_results = [run_graph_case(case) for case in cases]
    graph_summary = {
        'cases': len(graph_results),
        'doc_hit@k': bool_mean(graph_results, 'doc_hit'),
        'article_hit@k': bool_mean(graph_results, 'article_hit'),
        'fact_coverage': float_mean(graph_results, 'fact_coverage'),
        'display_citation_valid': bool_mean(graph_results, 'display_citation_valid'),
        'policy_success': bool_mean(graph_results, 'policy_success'),
        'hallucination_pass_rate': sum(1 for r in graph_results if r.get('hallucination_verdict') == 'pass') / len(graph_results),
        'avg_graph_ms': float_mean(graph_results, 'graph_ms'),
    }
else:
    graph_summary = {'skipped': True, 'reason': 'Set RUN_GRAPH=True to run full graph evaluation.'}

graph_summary

## Optional LLM Judge

Không bật mặc định. Chỉ nên dùng sau khi đã có rule metrics. LLM judge hữu ích cho `answer_relevance`, `legal_completeness`, và `faithfulness`, nhưng không thay thế được metadata/citation checks.

In [ ]:
LLM_JUDGE_PROMPT = '''
Bạn là evaluator độc lập cho hệ thống hỏi đáp pháp luật tiếng Việt.
Chấm câu trả lời theo context và expected facts.

Question: {question}
Expected facts: {expected_facts}
Answer: {answer}

Trả về JSON hợp lệ:
{{
  "answer_relevance": <0-1>,
  "legal_completeness": <0-1>,
  "faithfulness": <0-1>,
  "citation_support": <0-1>,
  "notes": "ngắn gọn"
}}
'''

def judge_with_llm(case: dict[str, Any], answer: str) -> dict[str, Any]:
    # Optional: uncomment when you really want LLM-as-judge.
    # from src.utils.llm_factory import get_model_with_fallback, parse_json_response
    # llm = get_model_with_fallback(purpose='evaluation_judge', json_mode=True)
    # prompt = LLM_JUDGE_PROMPT.format(
    #     question=case['question'],
    #     expected_facts=case.get('expected_facts', []),
    #     answer=answer,
    # )
    # response = llm.invoke(prompt)
    # return parse_json_response(response.content)
    raise NotImplementedError('LLM judge is intentionally disabled by default.')

## Save Report

In [17]:
timestamp = datetime.now().strftime('%Y%m%d_%H%M%S')
retrieval_failures = [
    r for r in retrieval_results
    if not (r['doc_hit'] and r['article_hit'] and r['clause_hit'] and r['point_hit'] and r['level_hit'])
]
report = {
    'created_at': datetime.now().isoformat(timespec='seconds'),
    'dataset': str(DATASET_PATH.relative_to(ROOT_DIR)),
    'top_k': TOP_K,
    'retrieval_summary': retrieval_summary,
    'answer_summary': answer_summary,
    'graph_summary': graph_summary if 'graph_summary' in globals() else {'skipped': True},
    'retrieval_results': retrieval_results,
    'answer_results': answer_results,
    'graph_results': graph_results if 'graph_results' in globals() else [],
}

json_path = REPORT_DIR / f'legal_qa_eval_{timestamp}.json'
json_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding='utf-8')

md_path = REPORT_DIR / f'legal_qa_eval_{timestamp}.md'
lines = [
    '# Legal QA Evaluation Report',
    '',
    f'- Created at: {report["created_at"]}',
    f'- Dataset: `{report["dataset"]}`',
    f'- Top K: {TOP_K}',
    '',
    '## Retrieval Summary',
    *[f'- {key}: {value:.4f}' if isinstance(value, float) else f'- {key}: {value}' for key, value in retrieval_summary.items()],
    '',
    '## Answer Summary',
    *[f'- {key}: {value:.4f}' if isinstance(value, float) else f'- {key}: {value}' for key, value in answer_summary.items()],
]
if retrieval_failures:
    lines.extend(['', '## Retrieval Failures'])
    for item in retrieval_failures[:20]:
        lines.append(f'- `{item["id"]}`: retrieved_docs={item["retrieved_doc_ids"]}, retrieved_articles={item["retrieved_articles"]}')

md_path.write_text('\n'.join(lines), encoding='utf-8')

print('Saved JSON report:', json_path)
print('Saved Markdown report:', md_path)

Saved JSON report: D:\source\Multi-Agent-RAG-for-Vietnamese-Legal-QA\eval_reports\legal_qa_eval_20260531_112538.json
Saved Markdown report: D:\source\Multi-Agent-RAG-for-Vietnamese-Legal-QA\eval_reports\legal_qa_eval_20260531_112538.md
